In [1]:
from ultralytics import YOLO
import albumentations as A

In [2]:
model = YOLO("yolo26n.pt")

In [4]:
custom_transform = [
    A.OneOf(
        [
            A.MotionBlur(blur_limit=7, p=1.0),
            A.MedianBlur(blur_limit=7, p=1.0),
            A.GaussianBlur(blur_limit=7, p=1.0)
        ],
        p=0.3
    ),

    A.OneOf(
        [
            A.GaussNoise(std_range=(0.1, 0.35), p=1.0),
            A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0)
        ],
        p=0.2
    ),

    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
    A.CoarseDropout(
        num_holes_range=(2, 8),
        hole_height_range=(8, 32), # Use integers for absolute pixels, or floats (0.0-1.0) for relative size
        hole_width_range=(8, 32),
        fill=0,
        p=0.5
    )
]
results = model.train(
    data="/home/ty21111/test_ML/janken_UseMLTools/collect_images/data.yaml",
    epochs=100,
    batch=16,
    workers=8,
    cache=True,
    device='cpu',
    augmentations=custom_transform,
    imgsz=480,
    #HueSaturationValueがhsv_h,s,vと、erasingがCoarseDropoutと被ってとんでもなくなる可能性があるため、パラメータを0にする
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    erasing=0.0
)


Ultralytics 8.4.51 🚀 Python-3.12.3 torch-2.12.0+cu130 CPU (13th Gen Intel Core i5-1335U)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[OneOf([
  MotionBlur(p=1.0, allow_shifted=True, angle_range=(0.0, 360.0), blur_limit=(3, 7), direction_range=(-1.0, 1.0)),
  MedianBlur(p=1.0, blur_limit=(3, 7)),
  GaussianBlur(p=1.0, blur_limit=(0, 7), sigma_limit=(0.5, 3.0)),
], p=0.3), OneOf([
  GaussNoise(p=1.0, mean_range=(0.0, 0.0), noise_scale_factor=1.0, per_channel=True, std_range=(0.1, 0.35)),
  ISONoise(p=1.0, color_shift=(0.01, 0.05), intensity=(0.1, 0.5)),
], p=0.2), CLAHE(p=0.5, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8)), RandomBrightnessContrast(p=0.5, brightness_by_max=True, brightness_limit=(-0.3, 0.3), contrast_limit=(-0.3, 0.3), ensure_safe_range=False), HueSaturationValue(p=0.5, hue_shift_limit=(-20.0, 20.0), sat_shift_limit=(-30.0, 30.0), val_shift_limit=(-20.0, 20.0)), CoarseDropout(p=0.5, fill=0.0, fill_mask=None, hole_height_range=(8, 

In [6]:
model_path = "runs/detect/YOLO26n_model/weights/best.pt"
onnx_path  = "runs/detect/YOLO26n_model/weights/best.onnx" 

onnx_model = YOLO(model_path)
onnx_model.export(format='onnx', imgsz=[480, 480])

Ultralytics 8.4.51 🚀 Python-3.12.3 torch-2.12.0+cu130 CPU (13th Gen Intel Core i5-1335U)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26n summary (fused): 122 layers, 2,375,421 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from 'runs/detect/YOLO26n_model/weights/best.pt' with input shape (1, 3, 480, 480) BCHW and output shape(s) (1, 300, 6) (9.1 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/17.6 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.1/17.6 MB 2.8 MB/s eta 0:00:07
   ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.3/17.6 MB 4.5 MB/s eta 0:00:04
   ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.5/17.6 MB 4.9 MB/s eta 0:00:04
   ━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.8/17.6 MB 5.8 MB/s eta 0:00:03
   ━━╺━━━━━━━━━━━━━━━━━━━━

/home/ty21111/test_ML/.myenv/lib/python3.12/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: export success ✅ 13.7s, saved as 'runs/detect/YOLO26n_model/weights/best.onnx' (9.3 MB)

Export complete (14.3s)
Results saved to /home/ty21111/test_ML/janken_UseMLTools/collect_images/runs/detect/YOLO26n_model/weights/best.onnx
Predict:         yolo predict task=detect model=runs/detect/YOLO26n_model/weights/best.onnx imgsz=480 
Validate:        yolo val task=detect model=runs/detect/YOLO26n_model/weights/best.onnx imgsz=480 data=/home/ty21111/test_ML/janken_UseMLTools/collect_images/data.yaml  
Visualize:       https://netron.app


'runs/detect/YOLO26n_model/weights/best.onnx'